In [11]:
# -- import libareis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import defaultdict


# We will do additional feature engineering in this notebook

In [2]:
matches_detailed = pd.read_csv('matches_w_stats_with_team_ranks.csv')
matches_detailed

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,...,away_bonus_rate,away_pin_count,away_avg_opponent_rank,away_avg_point_diff,away_avg_points_scored,away_std_point_diff,home_team_id,away_team_id,home_team_rank,away_team_rank
0,280,141,2025-11-01,267.0,Raymond Adams,160.0,825.0,John Hildebrandt,98.0,False,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,24,67,72,76
1,281,157,2025-11-01,840.0,Jonathan Ley,46.0,1038.0,Vince Bouzakis,82.0,False,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,68,27,21,19
2,281,165,2025-11-01,841.0,Dylan Elmore,34.0,298.0,Dylan Evans,22.0,False,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,68,27,21,19
3,281,174,2025-11-01,842.0,Danny Wask,6.0,300.0,Luca Augustine,21.0,True,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,68,27,21,19
4,281,184,2025-11-01,1256.0,Daniel Williams,61.0,301.0,Chase Kranitz,38.0,False,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,68,27,21,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5627,547,157,2026-02-23,1448.0,Phil Cuttino,212.0,822.0,Raymond Cmil,205.0,False,...,0.000000,0.0,106.500000,-9.800000,1.800000,6.300794,37,66,78,71
5628,547,174,2026-02-23,399.0,Joshua Roe,237.0,814.0,Beau Lewis,158.0,False,...,0.166667,1.0,100.666667,-3.909091,4.636364,7.687061,37,66,78,71
5629,547,184,2026-02-23,400.0,Reed Douglass,72.0,815.0,Andrew Barford,162.0,True,...,0.181818,1.0,111.818182,-6.300000,6.800000,9.499123,37,66,78,71
5630,547,197,2026-02-23,911.0,Toler Hornick,207.0,816.0,Toby Schoffstall,74.0,False,...,0.250000,1.0,98.250000,-2.714286,4.142857,8.400680,37,66,78,71


In [5]:
inference_pipeline_db = pd.read_csv('inference_pipeline_db.csv')
inference_pipeline_db.query('wrestler_name == "Gable Porter"')

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff
370,Gable Porter,Virginia,23,34,[141],12,9,3,75.0,6,50.0,2,3,1,3,86.17,8.73,4.36,8.66


In [35]:
def add_historical_matchup_features(matches_df):
    """
    Add historical matchup features to matches dataframe using unique wrestler_ids.
    Handles duplicate names by using wrestler_id as the unique identifier.
    """

    print("="*80)
    print("ADDING HISTORICAL MATCHUP FEATURES (USING WRESTLER_ID)")
    print("="*80)

    df = matches_df.copy()
    df = df.sort_values('event_date').reset_index(drop=True)

    # Initialize columns
    df['wrestled_before'] = 0
    df['home_point_diff_rematches'] = 0.0
    df['home_pinned_away'] = 0

    # Dictionary to store matchup history using wrestler_id pairs
    # Key: tuple (home_id, away_id) sorted
    # Value: dict with history
    matchup_history = defaultdict(lambda: {
        'matches': [],  # list of match records
        'total_point_diff': 0,
        'point_diff_count': 0,
        'pin_count': 0,
        'last_date': None,
        'wrestler1_name': None,
        'wrestler2_name': None
    })

    # Also create a lookup by name for convenience (but history uses IDs)
    name_to_ids = defaultdict(set)

    # Helper function to extract point differential from result
    def extract_point_diff(result, win_type, home_won):
        if pd.isna(result) or 'FALL' in str(win_type):
            return None

        try:
            # Extract all numbers from result
            numbers = re.findall(r'\d+', str(result))
            if len(numbers) >= 2:
                score1 = int(numbers[0])
                score2 = int(numbers[1])

                # Calculate point differential from perspective of winner
                if 'TF' in str(win_type):
                    return 15 if home_won else -15
                else:
                    if home_won:
                        return score1 - score2
                    else:
                        return score2 - score1
        except:
            pass
        return None

    print("\n🔄 Processing matches chronologically...")

    for idx, row in df.iterrows():
        home_id = row['home_wrestler_id']
        away_id = row['away_wrestler_id']
        home_name = row['home_name']
        away_name = row['away_name']

        # Skip if either ID is missing
        if pd.isna(home_id) or pd.isna(away_id):
            continue

        # Store name-to-id mapping for later lookup
        name_to_ids[home_name].add(home_id)
        name_to_ids[away_name].add(away_id)

        # Create sorted key using IDs (ensures consistency regardless of home/away)
        matchup_key = tuple(sorted([home_id, away_id]))

        # Get history if exists
        history = matchup_history[matchup_key]

        # Store names for reference
        if history['wrestler1_name'] is None:
            history['wrestler1_name'] = home_name if home_id == matchup_key[0] else away_name
            history['wrestler2_name'] = away_name if home_id == matchup_key[0] else home_name

        # Set wrestled_before
        if len(history['matches']) > 0:
            df.loc[idx, 'wrestled_before'] = 1

            # Calculate average point differential from previous matchups
            if history['point_diff_count'] > 0:
                avg_point_diff = history['total_point_diff'] / history['point_diff_count']
                df.loc[idx, 'home_point_diff_rematches'] = round(avg_point_diff, 3)

            # Check if home has ever pinned away
            # Need to look at previous matches where home_id was the winner and pinned
            for prev_match in history['matches']:
                if prev_match['was_pin'] and prev_match['winner_id'] == home_id:
                    df.loc[idx, 'home_pinned_away'] = 1
                    break

        # Now process current match to update history for future matches
        winner_id = home_id if row['home_win'] else away_id

        # Extract point differential for this match
        point_diff = extract_point_diff(row['Result'], row['win_type'], row['home_win'])
        was_pin = 'FALL' in str(row['win_type'])

        # Store this match in history
        match_record = {
            'date': row['event_date'],
            'winner_id': winner_id,
            'winner_name': home_name if row['home_win'] else away_name,
            'point_diff': point_diff,
            'was_pin': was_pin,
            'home_id': home_id,
            'away_id': away_id,
            'dual_id': row['dual_id']
        }

        history['matches'].append(match_record)

        if point_diff is not None:
            history['total_point_diff'] += point_diff
            history['point_diff_count'] += 1

        if was_pin:
            history['pin_count'] += 1

        history['last_date'] = row['event_date']

    print(f"✓ Processed {len(df)} matches")
    print(f"✓ Tracked {len(matchup_history)} unique wrestler pairings")

    # Check for duplicate names
    print("\n" + "="*80)
    print("CHECKING FOR DUPLICATE NAMES")
    print("="*80)

    duplicates_found = False
    for name, ids in name_to_ids.items():
        if len(ids) > 1:
            duplicates_found = True
            print(f"⚠️ Duplicate name: '{name}' appears with multiple IDs: {sorted(ids)}")

    if not duplicates_found:
        print("✅ No duplicate names found in dataset")

    return df, dict(matchup_history), dict(name_to_ids)

# ============================================
# FUNCTION TO GET HISTORICAL FEATURES FOR A SPECIFIC MATCHUP
# ============================================

def get_matchup_history_by_ids(wrestler1_id, wrestler2_id, matchup_history):
    """
    Get historical matchup data between two wrestlers using their IDs.
    Returns dict with historical data.
    """
    matchup_key = tuple(sorted([wrestler1_id, wrestler2_id]))
    history = matchup_history.get(matchup_key, {})

    if not history or len(history.get('matches', [])) == 0:
        return {
            'total_matches': 0,
            'wrestler1_wins': 0,
            'wrestler2_wins': 0,
            'avg_point_diff_wrestler1': 0,
            'pins_by_wrestler1': 0,
            'pins_by_wrestler2': 0,
            'last_winner_id': None,
            'last_date': None,
            'matches': []
        }

    wrestler1_id = matchup_key[0]
    wrestler2_id = matchup_key[1]

    wrestler1_wins = 0
    wrestler2_wins = 0
    total_point_diff = 0
    point_diff_count = 0
    pins_by_1 = 0
    pins_by_2 = 0

    for match in history['matches']:
        if match['winner_id'] == wrestler1_id:
            wrestler1_wins += 1
            if match['was_pin']:
                pins_by_1 += 1
        else:
            wrestler2_wins += 1
            if match['was_pin']:
                pins_by_2 += 1

        if match['point_diff'] is not None:
            # Point diff is from perspective of winner
            if match['winner_id'] == wrestler1_id:
                total_point_diff += match['point_diff']
            else:
                total_point_diff -= match['point_diff']
            point_diff_count += 1

    avg_point_diff_wrestler1 = total_point_diff / point_diff_count if point_diff_count > 0 else 0

    return {
        'total_matches': len(history['matches']),
        'wrestler1_name': history.get('wrestler1_name'),
        'wrestler2_name': history.get('wrestler2_name'),
        'wrestler1_wins': wrestler1_wins,
        'wrestler2_wins': wrestler2_wins,
        'avg_point_diff_wrestler1': round(avg_point_diff_wrestler1, 3),
        'pins_by_wrestler1': pins_by_1,
        'pins_by_wrestler2': pins_by_2,
        'last_winner_id': history['matches'][-1]['winner_id'],
        'last_date': history['matches'][-1]['date'],
        'matches': history['matches']
    }

# ============================================
# FUNCTION TO GET MATCHUP HISTORY BY NAMES (HANDLES DUPLICATES)
# ============================================

def get_matchup_history_by_names(wrestler1_name, wrestler2_name, name_to_ids, matchup_history):
    """
    Get historical matchup data between two wrestlers using their names.
    Handles duplicate names by checking all possible ID combinations.
    Returns list of possible matchups (in case of duplicates).
    """
    ids1 = name_to_ids.get(wrestler1_name, set())
    ids2 = name_to_ids.get(wrestler2_name, set())

    if not ids1 or not ids2:
        return []

    results = []
    for id1 in ids1:
        for id2 in ids2:
            history = get_matchup_history_by_ids(id1, id2, matchup_history)
            if history['total_matches'] > 0:
                results.append({
                    'wrestler1_id': id1,
                    'wrestler2_id': id2,
                    'history': history
                })

    return results

# ============================================
# TEST CASES
# ============================================

# Load your matches_detailed dataframe
matches_detailed = pd.read_csv('matches_w_stats_with_team_ranks.csv', parse_dates=['event_date'])

# Add historical features
matches_detailed_updated, matchup_history, name_to_ids = add_historical_matchup_features(matches_detailed)

# Test Case 1: Ben Davino vs Drake Ayala
print("\n" + "="*80)
print("TEST CASE 1: Ben Davino vs Drake Ayala")
print("="*80)

ben_drake = get_matchup_history_by_names('Ben Davino', 'Drake Ayala', name_to_ids, matchup_history)
if ben_drake:
    for matchup in ben_drake:
        print(f"\nBen Davino (ID: {matchup['wrestler1_id']}) vs Drake Ayala (ID: {matchup['wrestler2_id']})")
        print(f"Total matches: {matchup['history']['total_matches']}")
        print(f"Avg point diff (Ben): {matchup['history']['avg_point_diff_wrestler1']}")
        print(f"Pins by Ben: {matchup['history']['pins_by_wrestler1']}")
        print(f"Pins by Drake: {matchup['history']['pins_by_wrestler2']}")

        # Show all matches
        print("\nMatch details:")
        for m in matchup['history']['matches']:
            print(f"  {m['date']}: {m['winner_name']} won, point diff: {m['point_diff']}, pin: {m['was_pin']}")

# Test Case 2: Sergio Vega vs Brock Hardy
print("\n" + "="*80)
print("TEST CASE 2: Sergio Vega vs Brock Hardy")
print("="*80)

sergio_brock = get_matchup_history_by_names('Sergio Vega', 'Brock Hardy', name_to_ids, matchup_history)
if sergio_brock:
    for matchup in sergio_brock:
        print(f"\nSergio Vega (ID: {matchup['wrestler1_id']}) vs Brock Hardy (ID: {matchup['wrestler2_id']})")
        print(f"Total matches: {matchup['history']['total_matches']}")
        print(f"Avg point diff (Sergio): {matchup['history']['avg_point_diff_wrestler1']}")
        print(f"Pins by Sergio: {matchup['history']['pins_by_wrestler1']}")
        print(f"Pins by Brock: {matchup['history']['pins_by_wrestler2']}")

        # Show all matches
        print("\nMatch details:")
        for m in matchup['history']['matches']:
            print(f"  {m['date']}: {m['winner_name']} won, point diff: {m['point_diff']}, pin: {m['was_pin']}")

# Check for duplicate names like James Conway
print("\n" + "="*80)
print("CHECKING DUPLICATE NAME: James Conway")
print("="*80)

james_ids = name_to_ids.get('James Conway', set())
print(f"James Conway appears with IDs: {james_ids}")

for james_id in james_ids:
    # Find all opponents for this James Conway
    for key, history in matchup_history.items():
        if james_id in key:
            other_id = key[0] if key[1] == james_id else key[1]
            other_name = history['wrestler2_name'] if history['wrestler1_name'] == 'James Conway' else history['wrestler1_name']
            print(f"  James Conway (ID: {james_id}) vs {other_name} (ID: {other_id}): {len(history['matches'])} matches")

# Save updated dataframe
matches_detailed_updated.to_csv('matches_detailed_with_history.csv', index=False)
print("\n✅ Saved matches_detailed_with_history.csv")

# Save matchup history for later use in inference pipeline
import pickle
with open('matchup_history.pkl', 'wb') as f:
   pickle.dump({
       'matchup_history': matchup_history,
       'name_to_ids': name_to_ids
   }, f)
print("✅ Saved matchup_history.pkl for inference pipeline")

ADDING HISTORICAL MATCHUP FEATURES (USING WRESTLER_ID)

🔄 Processing matches chronologically...
✓ Processed 5632 matches
✓ Tracked 5561 unique wrestler pairings

CHECKING FOR DUPLICATE NAMES
⚠️ Duplicate name: 'Nathan Taylor' appears with multiple IDs: [168.0, 2001.0]
⚠️ Duplicate name: 'James Conway' appears with multiple IDs: [897.0, 2000.0]

TEST CASE 1: Ben Davino vs Drake Ayala

Ben Davino (ID: 443.0) vs Drake Ayala (ID: 196.0)
Total matches: 2
Avg point diff (Ben): -4.0
Pins by Ben: 0
Pins by Drake: 0

Match details:
  2025-11-16 00:00:00: Ben Davino won, point diff: 6, pin: False
  2026-02-06 00:00:00: Ben Davino won, point diff: 2, pin: False

TEST CASE 2: Sergio Vega vs Brock Hardy

Sergio Vega (ID: 592.0) vs Brock Hardy (ID: 217.0)
Total matches: 2
Avg point diff (Sergio): -11.0
Pins by Sergio: 0
Pins by Brock: 1

Match details:
  2025-11-16 00:00:00: Sergio Vega won, point diff: 11, pin: False
  2025-12-21 00:00:00: Sergio Vega won, point diff: None, pin: True

CHECKING DUPL

In [36]:
matches_detailed_updated.query('home_name == "Nick Feldman" or away_name == "Nick Feldman"')[["home_name", "home_wrestler_id", "away_name", "away_wrestler_id", "Result",
                                                                                              "wrestled_before","home_point_diff_rematches",	"home_pinned_away"]]

,home_name,home_wrestler_id,away_name,away_wrestler_id,Result,wrestled_before,home_point_diff_rematches,home_pinned_away
320,Ethan Vergara,869.0,Nick Feldman,451.0,LTF5 22 - 5 2:33,0,0.0,0
440,Nick Feldman,451.0,Christian Carroll,977.0,WDEC 7 - 3,0,0.0,0
495,Nick Feldman,451.0,Koy Hopke,158.0,WDEC 4 - 2,0,0.0,0
856,Nick Feldman,451.0,AJ Ferrari,224.0,WDEC 5 - 4,0,0.0,0
961,Nick Feldman,451.0,Brentan Simmerman,373.0,WTF5 20 - 4 4:34,0,0.0,0
1221,Nick Feldman,451.0,Daulton Mayer,1013.0,WTF5 17 - 1 3:36,0,0.0,0
1539,Nick Feldman,451.0,Isaac Trumble,244.0,LDEC 5 - 1,0,0.0,0
2035,Nick Feldman,451.0,Yonger Bastida,425.0,LDEC 4 - 2,0,0.0,0
2045,Nick Feldman,451.0,Kaden Darwin,615.0,WTF5 17 - 2 1:52,0,0.0,0
2854,Nick Feldman,451.0,Caleb Marzolino,957.0,WFALL 0:43,0,0.0,0


In [37]:
matchup_history

{(267.0,
  825.0): {'matches': [{'date': Timestamp('2025-11-01 00:00:00'),
    'winner_id': 825.0,
    'winner_name': 'John Hildebrandt',
    'point_diff': -1,
    'was_pin': False,
    'home_id': 267.0,
    'away_id': 825.0,
    'dual_id': 280}], 'total_point_diff': -1, 'point_diff_count': 1, 'pin_count': 0, 'last_date': Timestamp('2025-11-01 00:00:00'), 'wrestler1_name': 'Raymond Adams', 'wrestler2_name': 'John Hildebrandt'},
 (263.0,
  878.0): {'matches': [{'date': Timestamp('2025-11-01 00:00:00'),
    'winner_id': 263.0,
    'winner_name': 'Angelo Posada',
    'point_diff': -1,
    'was_pin': False,
    'home_id': 878.0,
    'away_id': 263.0,
    'dual_id': 283}], 'total_point_diff': -1, 'point_diff_count': 1, 'pin_count': 0, 'last_date': Timestamp('2025-11-01 00:00:00'), 'wrestler1_name': 'Angelo Posada', 'wrestler2_name': 'Kael Bennie'},
 (429.0,
  879.0): {'matches': [{'date': Timestamp('2025-11-01 00:00:00'),
    'winner_id': 879.0,
    'winner_name': 'Jack Forbes',
    'point_